# Lab 7 · Kiểm chứng quy tắc xử lý chuỗi

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · bài 7**

Chọn **Save a copy in Drive** trước khi sửa.

Lab dùng bảng nhỏ để kiểm quy tắc, rồi áp dụng lên `comments` và `amenities`
của Santiago, snapshot **29/06/2026**. Mục tiêu là tạo được kết quả có ý nghĩa,
không chỉ viết được một mẫu regex chạy thành công.

## Cách làm việc trong bài lab

- Điền các ô `TODO`; đọc kết quả và giải thích bằng dữ liệu gốc.
  `assert` kiểm một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với phần khởi động và bài có hướng dẫn, bạn nên tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Chuẩn hoá mà không tự làm mất thông tin; phân biệt lỗi đọc với dữ liệu đã thiếu.
2. Dùng quy tắc tìm, kiểm định dạng và trích xuất đúng mục đích.
3. Kiểm khớp nhầm trong văn bản thật.
4. Tính tỷ lệ tiện nghi theo chỗ ở, giữ đúng mẫu số.

## Phần 0 · Khởi động

Hai ô dưới dùng dữ liệu giả lập. Không tự gộp các cách viết ngoài quy tắc đã nêu.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
print("pandas:", pd.__version__)

In [ ]:
s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])

# TODO: bỏ khoảng trắng ở hai đầu và đổi sang chữ thường.
s_chuan = ...

assert s_chuan.tolist() == ["wifi", "wifi", "wifi!", "wi-fi"]
assert s_chuan.index.equals(s.index)
print(s_chuan.value_counts())

Hai giá trị cuối vẫn khác `wifi`. Nếu muốn gộp chúng, cần nêu thêm quy tắc, không mặc định xoá mọi dấu câu.

In [ ]:
s = pd.Series(["a.b", "axb", None])

# TODO: tạo mask tìm đúng dấu chấm; dòng thiếu không được chọn.
mask = ...

assert mask.tolist() == [True, False, False]
print(s[mask].tolist())

## Phần 1 · Đọc giá mà không che mất lỗi

Giá giả lập bằng USD/đêm. Quy tắc nguồn: `$` ở đầu, dấu phẩy phân cách hàng nghìn,
dấu chấm phân cách thập phân. Chỉ bỏ khoảng trắng ở hai đầu và các ký hiệu đã biết.
Chuỗi `liên hệ` chưa đọc được; `None` đã thiếu từ đầu.

In [ ]:
gia = pd.Series(["$1,200.00", " $850.00 ", None, "liên hệ"])

# TODO: chuyển sang số; chuỗi không đọc được nhận giá trị thiếu.
gia_so = ...
# TODO: chỉ đánh dấu chuỗi có dữ liệu nhưng chuyển đổi thất bại.
loi_doc = ...

assert gia_so.iloc[:2].tolist() == [1200.0, 850.0]
assert gia_so.iloc[2:].isna().all()
assert loi_doc.tolist() == [False, False, False, True]
assert gia.iloc[0] == "$1,200.00"
print(pd.DataFrame({"gia": gia, "gia_so": gia_so,
                    "loi_doc": loi_doc}))

### Kiểm cả chuỗi, tránh chỉ đọc một phần

Mẫu được cho dưới đây chỉ nhận số không âm: phần nguyên viết liền hoặc có dấu phẩy
sau mỗi ba chữ số; phần thập phân nếu có gồm đúng hai chữ số. Khoảng trắng ở hai đầu
được bỏ trước khi kiểm. Không dùng `extract` để kết luận cả chuỗi hợp lệ.

In [ ]:
gia_test = pd.Series(["$1,200.00", "$850", "$1,20.00",
                      "giá $120.00", "$120.00abc", None])
PAT_GIA = r"\$(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{2})?"

# TODO: mask cho biết cả chuỗi có đúng định dạng hay không.
hop_le = ...

assert hop_le.tolist() == [True, True, False, False, False, False]
print(pd.DataFrame({"gia": gia_test, "hop_le": hop_le}))

## Phần 2 · Làm sạch và kiểm quy tắc trên đánh giá thật

Bản đánh giá đầy đủ có cột `comments`. Không đọc `date` thành thời gian trong bài này;
xử lý thời gian thuộc Bài 8. Nếu có file snapshot ở thư mục `data`, ô dưới đọc bản đã lưu.

In [ ]:
URL_RV = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/data/reviews.csv.gz")
tep_rv = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
nguon_rv = tep_rv if tep_rv.exists() else URL_RV
rv = pd.read_csv(nguon_rv, usecols=["listing_id", "comments"])
print("Số đánh giá:", len(rv))

### Bước 1 · Khảo sát trước khi xử lý

Tạo `c` từ các đánh giá có nội dung, đo độ dài và xem một số đoạn ngắn.
Giữ chỉ mục để có thể quay lại đúng hàng trong `rv`.

In [ ]:
so_thieu = ...
c = ...
do_dai = ...

assert len(rv) == 690112
assert so_thieu == 36 and len(c) == 690076
assert c.index.equals(rv.index[rv["comments"].notna()])
assert do_dai.median() == 115.0
print(f"{so_thieu} đoạn thiếu; độ dài trung vị "
      f"{do_dai.median():.0f} ký tự")
print(c[do_dai < 3].head(5).tolist())

In [ ]:
# TODO: đếm các đoạn ngắn hơn 20 ký tự.
so_ngan = ...

assert so_ngan == 58987
print(f"{so_ngan:,}/{len(c):,} đoạn có độ dài < 20")
print(c[do_dai < 20].sample(n=5, random_state=7).tolist())

Độ dài là một thông tin khảo sát. Không mặc định mọi đoạn ngắn đều vô ích:
`Great host!` vẫn có thông tin. Quy tắc loại dữ liệu cần bám câu hỏi phân tích.

### Bước 2 · Thay đúng đoạn HTML đã biết

Nguồn lưu một số chỗ xuống dòng bằng `<br/>`. Thay đúng chuỗi này bằng một dấu cách,
giữ cột gốc để đối chiếu. Đây không phải bộ làm sạch mọi dạng HTML.

In [ ]:
so_br = ...
c_sach = ...

assert so_br == 116471
assert len(c_sach) == len(c) and c_sach.index.equals(c.index)
assert c_sach.str.contains("<br/>", regex=False, na=False).sum() == 0
assert c.str.contains("<br/>", regex=False, na=False).sum() == so_br
print(f"Đã thay <br/> trong {so_br:,} đoạn; giữ nguyên bản gốc")

### Bước 3 · Khớp cấu trúc có đồng nghĩa với lấy được địa điểm?

Quy tắc dưới lấy **từ đầu tiên** sau `cerca de`, `cerca de la`, `cerca del`,
`near`, `near the` (không phân biệt hoa thường). `\w+` khớp một dãy ký tự từ.
Chạy trên bảng nhỏ trước, rồi mới áp lên cả cột.

In [ ]:
PAT_GAN = r"\b(?:cerca de(?: la|l)?|near(?: the)?)\s+(\w+)"
mau = pd.Series(["near the metro", "cerca del parque",
                 "cerca de todo", "Casa centro"])
print(mau.str.extract(PAT_GAN, flags=re.I, expand=False))

`todo` nghĩa là “mọi thứ”, không phải tên địa điểm. Mẫu cũng chỉ lấy `Movistar`
trong `near Movistar Arena`, không lấy tên đầy đủ. Kết quả nên gọi là `tu_sau_gan`,
thay vì mặc định gọi là `dia_diem`.

In [ ]:
# TODO: lấy từ theo quy tắc trên, giữ chỉ mục và đếm số dòng khớp.
tu_sau_gan = ...
so_khop = ...

assert tu_sau_gan.index.equals(c_sach.index)
assert so_khop == 36613
print(tu_sau_gan.str.lower().value_counts().head(8))

**Đọc nguyên văn** để tìm ít nhất hai kết quả không phải tên địa điểm.
Số dòng khớp chỉ cho biết mức bao phủ của quy tắc, chưa đo độ chính xác.

In [ ]:
can_xem = tu_sau_gan.str.lower().isin(["todo", "muchos", "everything"])
print(c_sach[can_xem].sample(n=5, random_state=7).to_string(index=False))

Viết nhận xét của bạn ở đây: …

## Phần 3 · Tính tiện nghi theo chỗ ở

Các ô dưới dùng `amenities` của bảng chỗ ở đầy đủ. Mục tiêu là tỷ lệ chỗ ở
**liệt kê đúng mục `Wifi`**. Không suy ra mạng hoạt động tốt từ một mục được liệt kê.

In [ ]:
URL_LS = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2026-06-29/data/listings.csv.gz")
tep_ls = Path("data/listings_santiago_full_2026-06-29.csv.gz")
nguon_ls = tep_ls if tep_ls.exists() else URL_LS
ls = pd.read_csv(nguon_ls, usecols=["id", "amenities"]).set_index("id")
print("Số chỗ ở:", len(ls))

### Bước 4 · Đọc cấu trúc trước khi tách

Dùng `json.loads` trên từng ô, rồi `explode`. Giữ mã chỗ ở ở chỉ mục.
Chuỗi JSON có thể chứa dấu phẩy ngay trong một tên tiện nghi.

In [ ]:
# TODO: ds là Series các danh sách Python; muc là các mục sau khi tách, bỏ ô thiếu.
ds = ...
muc = ...

assert ds.index.equals(ls.index)
assert ds.map(type).eq(list).all()
assert ds.map(len).eq(0).sum() == 14
assert muc.eq("Wifi").sum() == 16772
print(muc.value_counts().head(5))

### Bước 5 · Giữ đúng mẫu số

Tạo một giá trị True/False cho mỗi chỗ ở, rồi tính tỷ lệ.
Danh sách rỗng là không có mục được liệt kê và vẫn nằm trong mẫu số.
Nếu một nguồn có `amenities` thiếu, đó là **chưa biết**, cần quy ước riêng.

In [ ]:
co_wifi = ...
ty_le_wifi = ...

assert len(co_wifi) == len(ls)
assert co_wifi.index.equals(ls.index)
assert co_wifi.sum() == 16772
assert abs(ty_le_wifi - 16772 / 18534) < 1e-12
print(f"{co_wifi.sum():,}/{len(ls):,} chỗ ở liệt kê Wifi "
      f"({ty_le_wifi:.1%})")

### Bước 6 · Phân biệt danh sách rỗng với dữ liệu thiếu

Bảng nhỏ giả lập có một danh sách lặp `Wifi`, một danh sách rỗng và một giá trị thiếu.
`[]` là biết không có mục được liệt kê; `None` là chưa biết danh sách.
Tính tỷ lệ chỗ ở có Wifi **trong nhóm đã có thông tin tiện nghi**.

In [ ]:
ds_nho = pd.Series([["Wifi", "Wifi", "Kitchen"],
                    ["Kitchen"], [], None],
                   index=["P1", "P2", "P3", "P4"])
so_da_biet = ...
ty_le_nho = ...

assert so_da_biet == 3
assert abs(ty_le_nho - 1 / 3) < 1e-12
print(f"Tỷ lệ đúng: {ty_le_nho:.1%}")

## Phần 4 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### Tự làm 1 · Kiểm một kết luận từ từ khoá

Một câu trả lời của AI: “Đếm các đánh giá chứa `wifi|internet` là đếm những khách hài lòng
về mạng.” Tạo mask từ khoá, đọc một mẫu **20 đoạn khớp**, rồi nêu các trường hợp làm
kết luận này sai. Viết lại kết luận sao cho phản ánh đúng phép đếm đã làm.

### Tự làm 2 · Trích diện tích có thêm cách viết

Quy tắc cần nhận `45 m2`, `30m²`, `45.5 m2`, `45,5 m²` và không nhận `30m2x`.
Trích và đưa về số, giữ nguyên phần còn lại của tên. Tự thêm các ví dụ không được khớp
trước khi nhờ AI sửa mẫu.

Nếu dùng AI, ghi công cụ, prompt chính và một trường hợp bạn đã tự kiểm.

In [ ]:
# Viết lời giải và cách kiểm chứng của bạn ở đây.

## Tóm tắt bài lab

- Quy tắc chuẩn hoá cần nêu rõ; giá chưa biết khác giá bằng 0.
- Tìm một phần khớp không thay thế kiểm cả định dạng.
- Khớp từ không xác minh ngữ nghĩa; xem nguyên văn để tìm lỗi.
- Tỷ lệ tiện nghi cần tính theo chỗ ở, kể cả danh sách rỗng và mục lặp.

Chạy **Restart & Run all** sau khi hoàn thành. Giải thích được một trường hợp
quy tắc khớp đúng và một trường hợp nó không đủ để kết luận.

**Tài liệu:** [McKinney §7.4](https://wesmckinney.com/book/data-cleaning#string-manipulation),
[pandas: dữ liệu chuỗi](https://pandas.pydata.org/docs/user_guide/text.html).

**Bài sau:** xử lý dữ liệu thời gian.